# Resume → Job Recommendation System

Pipeline:

```
Resume PDF → spaCy → Extract Candidate Skills → Sentence-BERT Normalization
   → Canonical Resume Skills → [Jaccard Skill Matching + Sentence-BERT Similarity]
   → Hybrid Similarity Score → Rank All Jobs → Top 3 Jobs
   → Matched Skills / Missing Skills
```

**What you need before running:** a pandas DataFrame `jobs_df` with a `job_title` column and a `skills` column
(comma-separated string, or a Python list of strings, per row). Replace the demo DataFrame in Step 1 with your own
(e.g. `pd.read_csv('your_jobs.csv')`).

In [1]:
!pip install requests


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import requests
import json

API_KEY = "8edd2b12-cf22-4cdc-89f3-fe5295c04f52"

url = f"https://jooble.org/api/{API_KEY}"

payload = {
    "keywords": "Machine Learning Engineer",
    "location": "Egypt"
}

response = requests.post(url, json=payload)

print(response.status_code)

200


In [12]:
data = response.json()

print(data.keys())

dict_keys(['totalCount', 'jobs'])


In [13]:
import pandas as pd

jobs = data["jobs"]

df = pd.DataFrame(jobs)

df.head()

,title,location,snippet,salary,source,type,link,company,updated,id
0,Data Scientist,Egypt,&nbsp;...will play a key role in leveraging da...,,swooped.co,,https://jooble.org/jdp/-5230836341694533438,Nawy Real Estate,2025-08-13T00:00:00.0000000,-5230836341694533438
1,Data Engineer,Egypt,&nbsp;...Proptech is in search of a highly mot...,,swooped.co,,https://jooble.org/jdp/1134570916703825292,Nawy Real Estate,2025-08-15T00:00:00.0000000,1134570916703825292
2,Senior Voice & Contact Center Engineer,Egypt,&nbsp;...We are looking for a seasoned Senior ...,,swooped.co,,https://jooble.org/jdp/-2607849692736859441,Nawy Real Estate,2025-08-13T00:00:00.0000000,-2607849692736859441
3,Senior Pre-Sales Engineer,Egypt,"&nbsp;..., high-impact digital infrastructures...",,hireskys.com,Temporary,https://jooble.org/jdp/2898285133897126864,Master Works,2026-07-08T02:57:06.4800000,2898285133897126864
4,Senior Data Science and AI Engineer,Egypt,&nbsp;...We are looking for a Senior AI <b>Eng...,,swooped.co,,https://jooble.org/jdp/-7678302189138409624,Nawy Real Estate,2025-08-13T00:00:00.0000000,-7678302189138409624


In [14]:
df = df[[
    "title",
    "company",
    "location",
    "link",
    "snippet"
]]

df.head()

,title,company,location,link,snippet
0,Data Scientist,Nawy Real Estate,Egypt,https://jooble.org/jdp/-5230836341694533438,&nbsp;...will play a key role in leveraging da...
1,Data Engineer,Nawy Real Estate,Egypt,https://jooble.org/jdp/1134570916703825292,&nbsp;...Proptech is in search of a highly mot...
2,Senior Voice & Contact Center Engineer,Nawy Real Estate,Egypt,https://jooble.org/jdp/-2607849692736859441,&nbsp;...We are looking for a seasoned Senior ...
3,Senior Pre-Sales Engineer,Master Works,Egypt,https://jooble.org/jdp/2898285133897126864,"&nbsp;..., high-impact digital infrastructures..."
4,Senior Data Science and AI Engineer,Nawy Real Estate,Egypt,https://jooble.org/jdp/-7678302189138409624,&nbsp;...We are looking for a Senior AI <b>Eng...


## Step 0 — Install dependencies (run once per Colab session)

In [15]:
!pip install -q spacy sentence-transformers pdfplumber scikit-learn pandas numpy
!python -m spacy download en_core_web_sm -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 62.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [16]:
!pip uninstall -y pillow
!pip install -U pillow

Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
  Using cached pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (9.1 kB)
Using cached pillow-12.3.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (6.9 MB)


In [17]:
import re
import numpy as np
import pandas as pd
import spacy
import pdfplumber
from spacy.matcher import PhraseMatcher
from sentence_transformers import SentenceTransformer, util
from google.colab import files


## Step 1 — Your jobs DataFrame

Replace this with your real data, e.g.:
```python
jobs_df = pd.read_csv('jobs.csv')   # must have 'job_title' and 'skills' columns
```
`skills` can be a comma-separated string like `"Python, SQL, Docker"` or already a Python list.

In [18]:
from google.colab import files
import pandas as pd

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

jobs_df = pd.read_csv(file_name)

jobs_df.head()

Saving job_dataset.csv to job_dataset (1).csv


,JobID,Title,ExperienceLevel,YearsOfExperience,Skills,Responsibilities,Keywords
0,NET-F-001,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Framework; .NET Core f...,Assist in coding and debugging applications; L...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
1,NET-F-002,.NET Developer,Fresher,0-1,C#; .NET Framework basics; ASP.NET; Razor; HTM...,Write simple C# programs under guidance; Suppo...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...
2,NET-F-003,.NET Developer,Fresher,0-1,C#; VB.NET basics; .NET Core; ASP.NET MVC; HTM...,Contribute to development of small modules; As...,.NET; C#; ASP.NET MVC; SQL Server; Entity Fram...
3,NET-F-004,.NET Developer,Fresher,0-1,C#; .NET Framework; ASP.NET basics; SQL Server...,Support in software design documentation; Assi...,.NET; C#; SQL Server; Entity Framework; ASP.NET
4,NET-F-005,.NET Developer,Fresher,0-1,C#; ASP.NET; MVC; Entity Framework basics; SQL...,Learn to design and build ASP.NET applications...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...


In [19]:
jobs_df = jobs_df.drop(columns=["JobID"])
jobs_df = jobs_df.drop(columns=["ExperienceLevel"])
jobs_df = jobs_df.drop(columns=["Responsibilities"])
jobs_df = jobs_df.drop(columns=["YearsOfExperience"])





In [20]:
jobs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1068 entries, 0 to 1067
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Title     1067 non-null   object
 1   Skills    1068 non-null   object
 2   Keywords  1068 non-null   object
dtypes: object(3)
memory usage: 25.2+ KB


In [21]:
import pandas as pd

jobs_df = jobs_df.drop_duplicates(subset=["Title"])

In [22]:
# ---- DEMO DATA: replace with your own jobs_df ----


def parse_skills(s):
    """Turn a skills cell (string or list) into a clean list of skill strings."""
    if isinstance(s, list):
        return [str(x).strip() for x in s if str(x).strip()]
    return [x.strip() for x in str(s).split(',') if x.strip()]

jobs_df['skills_list'] = jobs_df['Keywords'].apply(parse_skills)
jobs_df


,Title,Skills,Keywords,skills_list
0,.NET Developer,C#; VB.NET basics; .NET Framework; .NET Core f...,.NET; C#; ASP.NET MVC; Entity Framework; SQL S...,[.NET; C#; ASP.NET MVC; Entity Framework; SQL ...
20,AI Engineer - Fresher,Python; Java; C++; R; Supervised/Unsupervised ...,AI; ML; Fresher; Entry-Level; Python; Deep Lea...,[AI; ML; Fresher; Entry-Level; Python; Deep Le...
29,AI Engineer - Experienced,Python; Java; C++; R; Deep Learning; Reinforce...,AI; ML; Mid-Senior Level; Deep Learning; NLP; ...,[AI; ML; Mid-Senior Level; Deep Learning; NLP;...
36,NaN,Python; Java; C++; R; Deep Learning; NLP; Comp...,AI; ML; Senior-Level; Deep Learning; NLP; Clou...,[AI; ML; Senior-Level; Deep Learning; NLP; Clo...
40,AI Prompt Engineer,Python basics; JavaScript basics; Intro to NLP...,Prompt Engineering; NLP; Python; TensorFlow; P...,[Prompt Engineering; NLP; Python; TensorFlow; ...
...,...,...,...,...
1025,UX Consultant,Figma; Sketch; Design Thinking; User Research;...,Design Thinking; User Research; Agile; Usabili...,[Design Thinking; User Research; Agile; Usabil...
1026,Staff UX Designer,Figma; InVision; Design Systems; User Research...,Design Systems; A/B Testing; Figma; User Resea...,[Design Systems; A/B Testing; Figma; User Rese...
1027,UX Design Lead,Leadership; Figma; Design Systems; Interaction...,Leadership; Accessibility; Design Systems; Fig...,[Leadership; Accessibility; Design Systems; Fi...
1028,Vibe Coder,Python basics; JavaScript basics; HTML basics;...,AI-assisted coding; Python; JavaScript; React;...,[AI-assisted coding; Python; JavaScript; React...


## Step 2 — Upload the candidate's resume (PDF)

In [23]:
uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]
print(f"Uploaded: {pdf_path}")


Saving Rataj_Adel_El-Sayed_Hassan__Resume .pdf to Rataj_Adel_El-Sayed_Hassan__Resume  (1).pdf
Uploaded: Rataj_Adel_El-Sayed_Hassan__Resume  (1).pdf


In [24]:
def extract_text_from_pdf(path: str) -> str:
    """Extract raw text from a PDF resume."""
    text = ""
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            page_text = page.extract_text()
            if page_text:
                text += page_text + "\n"
    return text

resume_text = extract_text_from_pdf(pdf_path)
print(resume_text[:6000])


Rataj Adel El-Sayed Hassan
AI & Data Science Student | ML & AI Engineer
rouadel30@gmail.com 01118873081 Beni Suef, Egypt
portfolio-website-phi-eight-46.vercel.app/ acesse.one/ds0rbxo github.com/rataj-adel
Objective
AI & Data Science student passionate about technology, data, and problem-solving. Seeking opportunities in
AI, Data Science and ML to apply my skills, contribute to impactful projects, and continue growing
professionally.
Education
B.Sc. in Artificial Intelligence and Data Science, Beni Suef National University Beni Suef, Egypt
Faculty of Computers and Artificial Intelligence
Expected Graduation: 2027
•
Technical Skills
Programming Data Science
Python, Java, C++ Pandas, NumPy, Matplotlib, Data Analysis, Machine
Learning, Deep Learning basics
AI Concepts
LLMs, Model Training, Prediction, Data Processing Web Development
HTML, CSS, JavaScript (Basics)
Tools
Git, GitHub, VS Code, Excel Other
Problem Solving, Team Collaboration, APIs & JSON
Basics
Experiences
Technical Experience

## Step 3 — spaCy: extract candidate skill phrases

We build a skill vocabulary from every job posting's `skills_list`, use a spaCy `PhraseMatcher` to catch exact
mentions, and also pull noun chunks / entities as looser candidate phrases (to catch wording that doesn't
exactly match the vocab, e.g. "ML" for "Machine Learning") — those get resolved in the next step by SBERT.

In [25]:
nlp = spacy.load('en_core_web_sm')

# Canonical skill vocabulary = union of all skills across all job postings
all_skills = sorted(set(skill for skills in jobs_df['skills_list'] for skill in skills))
print(f"{len(all_skills)} unique skills in job vocabulary")

def build_matcher(nlp, skills_vocab):
    matcher = PhraseMatcher(nlp.vocab, attr='LOWER')
    patterns = [nlp.make_doc(skill) for skill in skills_vocab]
    matcher.add('SKILLS', patterns)
    return matcher

matcher = build_matcher(nlp, all_skills)

def extract_exact_skill_matches(text, nlp, matcher):
    doc = nlp(text)
    matches = matcher(doc)
    return sorted({doc[start:end].text for match_id, start, end in matches})

def extract_candidate_phrases(text, nlp, matcher):
    """Broad candidate phrase extraction: exact vocab matches + noun chunks + relevant entities."""
    doc = nlp(text)
    phrases = set(extract_exact_skill_matches(text, nlp, matcher))

    for chunk in doc.noun_chunks:
        cleaned = chunk.text.strip()
        if cleaned and 1 <= len(cleaned.split()) <= 4:
            phrases.add(cleaned)

    for ent in doc.ents:
        if ent.label_ in ('ORG', 'PRODUCT', 'LANGUAGE', 'WORK_OF_ART'):
            phrases.add(ent.text.strip())

    return sorted(p for p in phrases if p)

candidate_phrases = extract_candidate_phrases(resume_text, nlp, matcher)
print(f"{len(candidate_phrases)} candidate phrases extracted")
candidate_phrases[:1000]


205 unique skills in job vocabulary
105 candidate phrases extracted


['03/2024',
 '07/2024\nRecruited members',
 'AI',
 'AI & Data Science',
 'AI Concepts\nLLMs',
 'AI Engineer\nrouadel30@gmail.com',
 'AI Trainee',
 'APIs',
 'Adaptability Analytical Thinking\nFast',
 'Applied data analysis',
 'Artificial Intelligence',
 'Artificial Intelligence and Data Science, Beni Suef National University Beni Suef',
 'Basics',
 'Beni Suef',
 'C++ Pandas',
 'CSS',
 'Clear',
 'Clear & Active',
 'Computers',
 'DEPI',
 'Data Analysis',
 'Data Science',
 'Data Science and',
 'Digital',
 'Digital Egypt',
 'Digital Egypt Pioneers Initiative',
 'EDA',
 'Education\nB.Sc',
 'Egypt',
 'Egypt\nFaculty',
 'Excel',
 'Excel Other\nProblem Solving',
 'Expected Graduation',
 'GDSC Beni Suef Branch',
 'GitHub',
 'HR',
 'HR & PR',
 'House',
 'IEEE Beni Suef\nAssisted',
 'JSON',
 'Java',
 'JavaScript',
 'Leadership Time Management\nLeads',
 'Linear Regression',
 'ML',
 'ML & AI Engineer',
 'Machine\nLearning',
 'Matplotlib',
 'Model Training',
 'Negotiation & Persuasion Stress Manageme

## Step 4 — Sentence-BERT normalization → Canonical Resume Skills

Each candidate phrase is embedded and matched to the closest skill in the job vocabulary. Phrases whose best
match is below `threshold` are dropped (they're probably not real skills, e.g. "the team").

In [26]:
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

skill_vocab_embeddings = sbert_model.encode(all_skills, convert_to_tensor=True)

def normalize_to_canonical_skills(phrases, vocab, vocab_embeddings, model, threshold=0.60):
    """Map free-text candidate phrases to canonical job-vocabulary skills via SBERT similarity."""
    canonical = set()
    mapping = {}
    if not phrases:
        return [], {}

    phrase_embeddings = model.encode(phrases, convert_to_tensor=True)
    cos_scores = util.cos_sim(phrase_embeddings, vocab_embeddings)

    for i, phrase in enumerate(phrases):
        scores = cos_scores[i].cpu().numpy()
        best_idx = int(np.argmax(scores))
        best_score = float(scores[best_idx])
        if best_score >= threshold:
            canonical_skill = vocab[best_idx]
            canonical.add(canonical_skill)
            mapping[phrase] = (canonical_skill, round(best_score, 3))

    return sorted(canonical), mapping

canonical_resume_skills, skill_mapping = normalize_to_canonical_skills(
    candidate_phrases, all_skills, skill_vocab_embeddings, sbert_model, threshold=0.60
)

print("Canonical Resume Skills:")
canonical_resume_skills


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Canonical Resume Skills:


['Data Science Intern; Python; R; SQL; Machine Learning; Data Visualization; Data Analysis',
 'Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL',
 'Data Science; AI; Machine Learning; Python; R; SQL; Big Data; Leadership',
 'Java; Core Java; OOP; Collections; Git; Teamwork',
 'Lead ML/AI Engineer; Python; R; Java; ML/DL; MLOps; Big Data; Cloud Deployment; Leadership']

In [27]:
# Optional: inspect how raw phrases were normalized to canonical skills
pd.DataFrame(
    [(phrase, canon, score) for phrase, (canon, score) in skill_mapping.items()],
    columns=['resume_phrase', 'canonical_skill', 'similarity']
).sort_values('similarity', ascending=False).reset_index(drop=True)


,resume_phrase,canonical_skill,similarity
0,AI & Data Science,Data Science; AI; Machine Learning; Python; R;...,0.731
1,Data Science,Data Science Intern; Python; R; SQL; Machine L...,0.677
2,ML & AI Engineer,Lead ML/AI Engineer; Python; R; Java; ML/DL; M...,0.674
3,Data Science and,Data Science; AI; Machine Learning; Big Data; ...,0.651
4,Java,Java; Core Java; OOP; Collections; Git; Teamwork,0.625


## Step 5 — Hybrid similarity: Jaccard + Sentence-BERT

* **Jaccard** = overlap of exact canonical skill sets → rewards exact skill matches.
* **SBERT similarity** = cosine similarity between mean-pooled embeddings of each skill set → rewards
  semantic closeness even if the exact skill words differ.
* **Hybrid score** = weighted average of the two (tune `w_jaccard` / `w_sbert` to taste).

In [28]:
def jaccard_similarity(set_a, set_b):
    set_a, set_b = set(set_a), set(set_b)
    if not set_a and not set_b:
        return 0.0
    union = set_a | set_b
    return len(set_a & set_b) / len(union) if union else 0.0

def sbert_set_similarity(skills_a, skills_b, model):
    if not skills_a or not skills_b:
        return 0.0
    emb_a = model.encode(skills_a, convert_to_tensor=True).mean(dim=0)
    emb_b = model.encode(skills_b, convert_to_tensor=True).mean(dim=0)
    return float(util.cos_sim(emb_a, emb_b))

def hybrid_score(candidate_skills, job_skills, model, w_jaccard=0.5, w_sbert=0.5):
    jac = jaccard_similarity(candidate_skills, job_skills)
    sbert_sim = sbert_set_similarity(candidate_skills, job_skills, model)
    return w_jaccard * jac + w_sbert * sbert_sim, jac, sbert_sim


## Step 6 — Rank all jobs, return Top 3 with matched / missing skills

In [29]:
def recommend_top_jobs(candidate_skills, jobs_df, model, top_n=3, w_jaccard=0.5, w_sbert=0.5):
    rows = []
    for _, row in jobs_df.iterrows():
        job_skills = row['skills_list']
        score, jac, sbert_sim = hybrid_score(candidate_skills, job_skills, model, w_jaccard, w_sbert)
        matched = sorted(set(candidate_skills) & set(job_skills))
        missing = sorted(set(job_skills) - set(candidate_skills))
        rows.append({
            'job_title': row['Title'],
            'hybrid_score': round(score, 4),
            'jaccard': round(jac, 4),
            'sbert_similarity': round(sbert_sim, 4),
            'matched_skills': matched,
            'missing_skills': missing
        })
    result_df = pd.DataFrame(rows).sort_values('hybrid_score', ascending=False).reset_index(drop=True)
    return result_df.head(top_n)

top3 = recommend_top_jobs(canonical_resume_skills, jobs_df, sbert_model, top_n=3)
top3


,job_title,hybrid_score,jaccard,sbert_similarity,matched_skills,missing_skills
0,Data Science Team Lead,0.5661,0.2,0.9322,[Data Science; AI; Machine Learning; Big Data;...,[]
1,Lead Data Scientist,0.5654,0.2,0.9307,[Data Science; AI; Machine Learning; Python; R...,[]
2,Lead ML/AI Engineer,0.5188,0.2,0.8376,[Lead ML/AI Engineer; Python; R; Java; ML/DL; ...,[]


In [30]:
for i, row in top3.iterrows():
    print(f"#{i+1}: {row['job_title']}  —  Hybrid score: {row['hybrid_score']}")
    print(f"    Jaccard: {row['jaccard']}   |   SBERT similarity: {row['sbert_similarity']}")
    print(f"    Matched skills:  {', '.join(row['matched_skills']) if row['matched_skills'] else 'None'}")
    print(f"    Missing skills:  {', '.join(row['missing_skills']) if row['missing_skills'] else 'None'}")
    print()


#1: Data Science Team Lead  —  Hybrid score: 0.5661
    Jaccard: 0.2   |   SBERT similarity: 0.9322
    Matched skills:  Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL
    Missing skills:  None

#2: Lead Data Scientist  —  Hybrid score: 0.5654
    Jaccard: 0.2   |   SBERT similarity: 0.9307
    Matched skills:  Data Science; AI; Machine Learning; Python; R; SQL; Big Data; Leadership
    Missing skills:  None

#3: Lead ML/AI Engineer  —  Hybrid score: 0.5188
    Jaccard: 0.2   |   SBERT similarity: 0.8376
    Matched skills:  Lead ML/AI Engineer; Python; R; Java; ML/DL; MLOps; Big Data; Cloud Deployment; Leadership
    Missing skills:  None



In [36]:
# ============================================================
# Interview Question Generator
# Part 1
# ============================================================

import re
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ============================================================
# اختيار الجهاز
# ============================================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Using Device: {DEVICE}")

# ============================================================
# تحميل الموديل
# ============================================================

MODEL_NAME = "google/flan-t5-large"
# لو الرام قليلة استخدمي:
# MODEL_NAME = "google/flan-t5-base"

print("Loading Interview Question Generator...")

question_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

question_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32
)

question_model.to(DEVICE)
question_model.eval()

print("Model Loaded Successfully")

# ============================================================
# Prompt Builder
# ============================================================

def build_prompt(
    job_title,
    matched_skills,
    missing_skills,
    candidate_skills,
    num_questions=5,
    question_type="technical"
):

    matched = matched_skills[:5] if matched_skills else []
    missing = missing_skills[:5] if missing_skills else []
    candidate = candidate_skills[:10] if candidate_skills else []

    if question_type == "technical":

        prompt = f"""
You are a Senior Technical Interviewer.

Generate exactly {num_questions} UNIQUE technical interview questions.

Job Title:
{job_title}

Candidate Skills:
{', '.join(candidate)}

Strong Skills:
{', '.join(matched)}

Missing Skills:
{', '.join(missing)}

Rules:

- Questions must be specific.
- Questions must NOT repeat.
- Focus on practical experience.
- Include scenario-based questions.
- Include one question about missing skills.
- Output ONLY the questions.
- Number each question.

Example:

1. ...
2. ...
"""

    elif question_type == "behavioral":

        prompt = f"""
You are an HR Interviewer.

Generate exactly {num_questions} behavioral interview questions.

Job:
{job_title}

Candidate Skills:
{', '.join(candidate)}

Focus on:

- teamwork
- leadership
- communication
- problem solving
- adaptability

Output only numbered questions.
"""

    else:

        prompt = f"""
Generate exactly {num_questions} interview questions.

Half technical.

Half behavioral.

Job:
{job_title}

Candidate Skills:
{', '.join(candidate)}

Matched Skills:
{', '.join(matched)}

Missing Skills:
{', '.join(missing)}

Output only numbered questions.
"""

    return prompt
    # ============================================================
# Part 2
# توليد الأسئلة وتنظيفها
# ============================================================

def clean_questions(text):
    """
    استخراج الأسئلة من النص الناتج
    """

    questions = []

    lines = text.split("\n")

    for line in lines:

        line = line.strip()

        if not line:
            continue

        # إزالة الترقيم
        line = re.sub(r"^\d+[\.\)]\s*", "", line)

        # إزالة الـ bullets
        line = re.sub(r"^[-*]\s*", "", line)

        # إزالة كلمة Question
        line = re.sub(
            r"^Question\s*\d*[:\-]?\s*",
            "",
            line,
            flags=re.IGNORECASE
        )

        line = line.strip()

        # تجاهل السطور القصيرة
        if len(line) < 20:
            continue

        # إضافة علامة استفهام إذا الموديل نسيها
        if not line.endswith("?"):
            line += "?"

        if line not in questions:
            questions.append(line)

    return questions


# ============================================================
# توليد الأسئلة
# ============================================================

def generate_interview_questions(
        job_title,
        matched_skills,
        missing_skills,
        candidate_skills,
        num_questions=5,
        question_type="technical"
):

    prompt = build_prompt(
        job_title,
        matched_skills,
        missing_skills,
        candidate_skills,
        num_questions,
        question_type
    )

    inputs = question_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():

        outputs = question_model.generate(

            **inputs,

            max_new_tokens=250,

            temperature=1.0,

            do_sample=True,

            top_p=0.95,

            top_k=50,

            repetition_penalty=1.2,

            no_repeat_ngram_size=3
        )

    generated_text = question_tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    questions = clean_questions(generated_text)

    # إزالة التكرار النهائي
    unique_questions = []

    seen = set()

    for q in questions:

        key = q.lower().strip()

        if key not in seen:

            seen.add(key)

            unique_questions.append(q)

    # لو الموديل رجع أقل من المطلوب
    if len(unique_questions) < num_questions:

        fallback = generate_fallback_questions(
            job_title,
            matched_skills,
            missing_skills
        )

        for q in fallback:

            if q.lower() not in seen:

                unique_questions.append(q)

                seen.add(q.lower())

            if len(unique_questions) == num_questions:
                break

    return unique_questions[:num_questions]


# ============================================================
# Fallback Questions
# ============================================================

def generate_fallback_questions(
        job_title,
        matched_skills,
        missing_skills
):

    questions = []

    for skill in matched_skills:

        questions.append(
            f"Can you explain a real project where you used {skill} successfully?"
        )

        questions.append(
            f"What challenges have you faced while working with {skill}?"
        )

    for skill in missing_skills:

        questions.append(
            f"What is your current knowledge of {skill}, and how would you learn it quickly?"
        )

    questions.extend([

        f"Why are you interested in the {job_title} position?",

        "Tell me about your most challenging technical project.",

        "Describe a time when you had to solve a difficult problem under pressure.",

        "How do you stay updated with new technologies?",

        "How do you debug an application when you don't know the source of the bug?",

        "Tell me about a mistake you made in a project and what you learned from it."

    ])

    return questions
    # ============================================================
# Part 3
# Generate Questions for Recommended Jobs
# ============================================================

def generate_questions_for_jobs(
    top_jobs_data,
    num_questions=5,
    question_type="technical"
):
    """
    Generate interview questions for each recommended job.
    """

    all_results = []

    for job in top_jobs_data:

        result = {

            "job_title": job.get("job_title", "Unknown Job"),

            "match_score": job.get(
                "hybrid_score",
                job.get("score", 0)
            ),

            "matched_skills": job.get(
                "matched_skills",
                []
            ),

            "missing_skills": job.get(
                "missing_skills",
                []
            )
        }

        score = job.get("hybrid_score", 0)
        if score >= 0.80:
          current_num_questions = 8
        elif score >= 0.60:
          current_num_questions = 6
        else:
          current_num_questions = 5

        questions = generate_interview_questions(

            job_title=result["job_title"],

            matched_skills=result["matched_skills"],

            missing_skills=result["missing_skills"],

            candidate_skills=job.get(
                "candidate_skills",
                []
            ),

            num_questions=num_questions,

            question_type=question_type
        )

        result["interview_questions"] = questions

        all_results.append(result)

    return all_results


# ============================================================
# Display
# ============================================================

def display_questions(results):

    print("\n")
    print("=" * 100)
    print("AI INTERVIEW QUESTION GENERATOR".center(100))
    print("=" * 100)

    for idx, item in enumerate(results, start=1):

        print("\n")
        print("=" * 100)

        print(f"Job #{idx}")
        print(f"Title : {item['job_title']}")
        print(f"Match : {item['match_score']:.2%}")

        print("-" * 100)

        print("Matched Skills:")

        if item["matched_skills"]:
            print(", ".join(item["matched_skills"]))
        else:
            print("None")

        print()

        print("Missing Skills:")

        if item["missing_skills"]:
            print(", ".join(item["missing_skills"]))   # ✅ تم إصلاح الخطأ
        else:
            print("None")

        print()

        print("Interview Questions")
        print("-" * 100)

        for i, q in enumerate(item["interview_questions"], start=1):

            print(f"{i}. {q}")

        print("=" * 100)





Using Device: cuda
Loading Interview Question Generator...


Loading weights:   0%|          | 0/558 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Loaded Successfully


In [40]:
results = generate_questions_for_jobs(
    top3.to_dict('records'), # Convert DataFrame to a list of dictionaries
    num_questions=5,
    question_type="mixed"
)

display_questions(results)



                                  AI INTERVIEW QUESTION GENERATOR                                   


Job #1
Title : Data Science Team Lead
Match : 56.61%
----------------------------------------------------------------------------------------------------
Matched Skills:
Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL

Missing Skills:
None

Interview Questions
----------------------------------------------------------------------------------------------------
1. How many different industries are represented in the company?
2. Can you explain a real project where you used Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL successfully?
3. What challenges have you faced while working with Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL?
4. Why are you interested in the Data Science Team Lead position?
5. Tell me about your most challenging technical project.


Job #2
Title : Lead Data Scientist
Match : 56.54%
----

In [32]:
display(top3.head())

,job_title,hybrid_score,jaccard,sbert_similarity,matched_skills,missing_skills
0,Data Science Team Lead,0.5661,0.2,0.9322,[Data Science; AI; Machine Learning; Big Data;...,[]
1,Lead Data Scientist,0.5654,0.2,0.9307,[Data Science; AI; Machine Learning; Python; R...,[]
2,Lead ML/AI Engineer,0.5188,0.2,0.8376,[Lead ML/AI Engineer; Python; R; Java; ML/DL; ...,[]


In [34]:
results = generate_questions_for_jobs(
    top3.to_dict('records'), # Convert DataFrame to a list of dictionaries
    num_questions=5,
    question_type="mixed"
)

display_questions(results)


                            🎯 أسئلة المقابلة المخصصة                            

الوظيفة #1: Data Science Team Lead
درجة التطابق: 56.61%

📌 المهارات المتطابقة:
   Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL

⚠️ المهارات الناقصة:
   ممتاز! لا توجد مهارات ناقصة

❓ أسئلة المقابلة المقترحة (5 سؤال):
   1. What type of job does the Data Science Team Lead position require?
   2. What are the best practices when using Data Science; AI; Machine Learning; Big Data; Leadership; Python; R; SQL in a professional environment?
   3. What interests you most about the Data Science Team Lead position?
   4. Describe a challenging project you worked on and how you overcame obstacles.
   5. Where do you see yourself in 5 years in your career as a Data Science Team Lead?

--------------------------------------------------------------------------------

الوظيفة #2: Lead Data Scientist
درجة التطابق: 56.54%

📌 المهارات المتطابقة:
   Data Science; AI; Machine Learning; Python; R